# Morphological Image Processing and Color Segmentation

This notebook explores binary morphology, skin-lesion mask refinement, and HSV-based object segmentation with OpenCV.


In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


## Input data

The source images are stored in the repository's `data/` directory. Run this notebook from the repository root so all relative paths resolve correctly.


In [ ]:
# All input paths are relative to the repository root.
# Run the notebook from that directory so it works locally and on Colab.
assert DATA_DIR.exists(), "Missing data directory; run from the repository root."


## 1. Filling holes in binary objects


### 1.1 Read and display the image

Load `q1.png` as a grayscale image and inspect it.


In [ ]:
img = cv2.imread(str(DATA_DIR / "q1.png"), cv2.IMREAD_GRAYSCALE)
if img is None:
    raise FileNotFoundError(DATA_DIR / "q1.png")

plt.imshow(img, cmap="gray")
plt.title("Grayscale image")
plt.axis("off")
plt.show()


### 1.2 Remove black holes with morphology

Use morphological operations to fill the black regions inside the white circular objects.


In [ ]:
# Conver to binary using thresholding
gray = img
nothing , bw = cv2.threshold(gray , 0, 255,cv2.THRESH_BINARY + cv2.THRESH_OTSU)
kernel1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (42, 42))
# just for test (30,30)
kernel2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (30, 30))
# Use closing to fill the black holes
bw_closed1 = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel1)
bw_closed2 = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel2)
# Show input and output images
# kernel1
plt.figure(figsize=(10,10))
plt.subplot(1, 2, 1)
plt.imshow(bw_closed1, cmap='gray')
plt.title(' kernel1 ')
plt.axis('off')
# kernel2
plt.subplot(1, 2, 2)
plt.imshow(bw_closed2, cmap='gray')
plt.title('kernel2')
plt.axis('off')
plt.show()

**Method.** The dark regions are relatively large holes, so small structuring elements cannot fill them completely. Morphological closing with a large elliptical kernel fills the holes while preserving the objects' overall circular form. Two kernel sizes are compared to illustrate the effect of the structuring element.


### 1.3 Compare and analyze the result

Display the binary input and processed result side by side.


In [ ]:
# Gray Image
plt.figure(figsize=(10,10))
plt.subplot(1, 2, 1)
plt.imshow(bw, cmap='gray')
plt.title('Gray Image ')
plt.axis('off')
# After Closing
plt.subplot(1, 2, 2)
plt.imshow(bw_closed1, cmap='gray')
plt.title('After Closing')
plt.axis('off')
plt.show()

**Analysis.** The original objects contain dark central holes. Closing fills these areas and produces solid white shapes. A kernel that is too small leaves part of a hole unfilled, while an excessively large kernel can merge nearby objects or distort their boundaries. An adaptive kernel or contour-based hole filling would be useful when hole sizes vary substantially.


## 2. Morphological processing of skin-lesion masks


### 2.1 Threshold the lesion images

Read the four lesion images, normalize their grayscale intensities, and apply a threshold of 0.25 so each lesion appears white against a black background.


In [ ]:
img_paths = [DATA_DIR / "melanome1.jpg", DATA_DIR / "melanome2.jpg",
             DATA_DIR / "melanome3.jpg", DATA_DIR / "melanome4.JPG"]
binary_images = []
T = 0.25

for path in img_paths:
    img_gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        raise FileNotFoundError(path)
    img_norm = img_gray.astype(np.float32) / 255.0
    binary_images.append((img_norm < T).astype(np.uint8) * 255)

plt.figure(figsize=(10, 5))
for i, binary_image in enumerate(binary_images, start=1):
    plt.subplot(1, 4, i)
    plt.imshow(binary_image, cmap="gray")
    plt.title(f"Binary {i}")
    plt.axis("off")
plt.tight_layout()
plt.show()


### 2.2 Improve connectivity in lesion 1

Remove small noise, connect nearby regions, and fill internal holes.


In [ ]:
# Select the first binary lesion mask
bw1 = binary_images[0]
bw1_bin = (bw1 > 0).astype(np.uint8)
# Remove small noise with opening
# Opening = erosion then dilation
se_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
bw1_open = cv2.morphologyEx(bw1_bin, cv2.MORPH_OPEN, se_open)
# Fill small gaps and connect components with closing
# Closing = dilation then erosion
se_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
bw1_close = cv2.morphologyEx(bw1_open, cv2.MORPH_CLOSE, se_close)
# Fill internal holes
h, w = bw1_close.shape
mask = np.zeros((h+2, w+2), np.uint8)
flood = bw1_close.copy()
cv2.floodFill(flood, mask, (0,0), 1)
flood_inv = 1 - flood
bw1_filled = bw1_close | flood_inv
# display result
plt.figure(figsize=(12,5))
plt.subplot(1,3,1); plt.imshow(bw1, cmap='gray');
plt.title("Original Binary 1"); plt.axis('off')
plt.subplot(1,3,2); plt.imshow(bw1_close*255, cmap='gray');
plt.title("Opening + Closing"); plt.axis('off')
plt.subplot(1,3,3); plt.imshow(bw1_filled*255, cmap='gray');
plt.title("Final Filled Lesion"); plt.axis('off')
plt.show()

**Method.** Opening with a 5×5 elliptical kernel removes small isolated structures and thin artifacts. Closing with a 9×9 elliptical kernel reconnects nearby lesion regions and fills narrow gaps. Flood filling then identifies and fills enclosed background regions, producing a solid lesion mask.


### 2.3 Remove hair and noise from lesion 2

Suppress thin hair-like structures, then recover the main lesion's approximate size.


In [ ]:
bw2 = binary_images[1]
# convert to 0/1
bw2_bin = (bw2 > 0).astype(np.uint8)
# Apply morphology to remove noise
# remove hairs and small noise
# hairs and small spots are removed by erosion.
se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
eroded = cv2.erode(bw2_bin, se, iterations=1)
# Dilate to restore the lesion after erosion
# dilation to recover the main lesion size after erosion
clean_lesion = cv2.dilate(eroded, se, iterations=1)
# display
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(bw2, cmap='gray')
plt.title('Original binary (lesion 2)')
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(eroded * 255, cmap='gray')
plt.title('After erosion (hairs removed)')
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(clean_lesion * 255, cmap='gray')
plt.title('After dilation (final lesion)')
plt.axis('off')
plt.show()

**Method.** Erosion with a 7×7 elliptical kernel removes thin lines and small foreground spots. Dilation with the same kernel restores the main lesion after erosion. Together, these steps form an opening operation that suppresses narrow artifacts while retaining the larger lesion region.


### 2.4 Fill the internal hole in lesion 3


In [ ]:
bw3 = binary_images[2]
# Convert to binary format (0 / 255)
bw3_bin = (bw3 > 0).astype(np.uint8) * 255
# Closing
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
bw3_closed = cv2.morphologyEx(bw3_bin,cv2.MORPH_CLOSE,kernel,iterations=1)
# Hole filling using Flood-Fill
flood = bw3_closed.copy()
h, w = flood.shape
mask = np.zeros((h + 2, w + 2), np.uint8)
# Fill the outer background
cv2.floodFill(flood, mask, (0, 0), 255)
# Invert flood-filled image to obtain internal holes
flood_inv = cv2.bitwise_not(flood)
# Combine closed lesion with filled holes
bw3_filled = cv2.bitwise_or(bw3_closed, flood_inv)
# display results
plt.figure(figsize=(14, 4))
# Original Binary
plt.subplot(1, 3, 1)
plt.imshow(bw3_bin, cmap='gray')
plt.title("Original Binary")
plt.axis('off')
# After Closing
plt.subplot(1, 3, 2)
plt.imshow(bw3_closed, cmap='gray')
plt.title("After Closing")
plt.axis('off')
# After Hole Filling
plt.subplot(1, 3, 3)
plt.imshow(bw3_filled, cmap='gray')
plt.title("After Hole Filling")
plt.axis('off')
plt.show()

**Method.** Closing first removes small gaps. Flood filling begins at the image border to mark the external background; inverting that result isolates enclosed holes. Combining those holes with the closed mask produces a solid lesion without relying on a dedicated hole-filling function.


### 2.5 Separate touching regions in lesion 4


In [ ]:
# Select the fourth binary lesion mask
bw4 = binary_images[3]
# convert to 0/1
bw4_bin = (bw4 > 0).astype(np.uint8)
# Erode the mask to separate touching regions
se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17,17))  # Large kernel used to separate the two regions
eroded = cv2.erode(bw4_bin, se, iterations=1)
# Label the separated regions
num_labels, labels = cv2.connectedComponents(eroded)
# Display the result
plt.figure(figsize=(10,4))
# display
plt.subplot(1,3,1)
plt.imshow(bw4_bin*255, cmap='gray')
plt.title("Original Binary (Lesion 4)")
plt.axis('off')
# After Erosion
plt.subplot(1,3,2)
plt.imshow(eroded*255, cmap='gray')
plt.title("After Erosion ")
plt.axis('off')
# Connected Components
plt.subplot(1,3,3)
plt.imshow(labels, cmap='gray')
plt.title("Connected Components")
plt.axis('off')
# displa
plt.show()

**Method.** Erosion with a 17×17 elliptical kernel breaks the narrow bridge between the two touching regions. Connected-component labeling then assigns a different label to each separated region. The kernel size is large enough to break the connection but may shrink the objects, so dilation could be applied afterward when boundary recovery is needed.


## 3. HSV clothing segmentation and recoloring

Create and refine a mask for the green garment in `q3.png`, then recolor the detected garment red.


### 3.1 Create the initial mask

Threshold the image in HSV color space and save the initial mask as `outputs/q3res01.jpg`.


In [ ]:
img_bgr = cv2.imread(str(DATA_DIR / "q3.png"))
if img_bgr is None:
    raise FileNotFoundError(DATA_DIR / "q3.png")
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
lower_cloth = np.array([40, 80, 80])
upper_cloth = np.array([80, 255, 255])
mask_initial = cv2.inRange(img_hsv, lower_cloth, upper_cloth)
cv2.imwrite(str(OUTPUT_DIR / "q3res01.jpg"), mask_initial)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
plt.title("Original image")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(mask_initial, cmap="gray")
plt.title("Initial mask")
plt.axis("off")
plt.tight_layout()
plt.show()


### 3.2 Refine the mask

Use closing and opening to fill small holes and remove isolated noise. Save the improved mask as `outputs/q3res02.jpg`.


In [ ]:
# convert mask to 0/1 if it is 0/255
mask_bin = (mask_initial > 0).astype(np.uint8)
#structuring element
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
# Closing
mask_closed = cv2.morphologyEx(mask_bin, cv2.MORPH_CLOSE, kernel, iterations=2)
# Opening
mask_opened = cv2.morphologyEx(mask_closed, cv2.MORPH_OPEN, kernel, iterations=1)
# final mask
mask_final = (mask_opened * 255).astype(np.uint8)
# Save the final mask
cv2.imwrite(str(OUTPUT_DIR / 'q3res02.jpg'), mask_final)
# Compare the initial and final masks
plt.figure(figsize=(10,4))
# display Initial mask
plt.subplot(1,2,1)
plt.imshow(mask_initial, cmap='gray')
plt.title('Initial mask')
plt.axis('off')
# display Improved mask
plt.subplot(1,2,2)
plt.imshow(mask_final, cmap='gray')
plt.title('Improved mask')
plt.axis('off')
plt.show()

**Method.** Closing fills small gaps inside the clothing region, and opening removes small isolated foreground areas. An elliptical 7×7 structuring element provides smooth boundaries and works well for the scale of the visible artifacts.


### 3.3 Recolor the garment

Keep the largest connected component, recolor that region red, and save the result as `outputs/q3res03.jpg`.


In [ ]:
img_bgr = cv2.imread(str(DATA_DIR / 'q3.png'))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
# mask_final
mask_bin = (mask_final > 0).astype(np.uint8)
# connected components
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)
largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
# mask
mask_refined = (labels == largest_label).astype(np.uint8)
mask_refined_255 = (mask_refined * 255).astype(np.uint8)
result_rgb = img_rgb.copy()
# red
red_color = np.array([255, 0, 0], dtype=np.uint8)
# just clothe
result_rgb[mask_refined == 1] = red_color
# save final pic
result_bgr = cv2.cvtColor(result_rgb, cv2.COLOR_RGB2BGR)
cv2.imwrite(str(OUTPUT_DIR / 'q3res03.jpg'), result_bgr)
# display result
plt.figure(figsize=(15,5))
plt.subplot(1,3,1)
plt.imshow(img_rgb)
plt.title('Original')
plt.axis('off')
plt.subplot(1,3,2)
plt.imshow(mask_refined_255, cmap='gray')
plt.title('Mask Clothes')
plt.axis('off')
# Display Final Colored Clothing
plt.subplot(1,3,3)
plt.imshow(result_rgb)
plt.title('Final Colored Clothing')
plt.axis('off')
plt.show()